In [ ]:
from clearml import Task

task = Task.init(
    project_name="Assignment 2 : Object Detection",
    task_name="SSD_MobileNet_300_Epochs"
)

In [ ]:
task.execute_remotely(queue_name='default')

### 42028 : Deep Learning and CNN
### Assignment 2

| Name | Swapnil Ajit Chhatre |
| :--- | :--- |
| Student ID | 25675238 |
| Tutorial | 04 (Wednesday 12.00 - 15.00) |
| Tutor | Mr. Deep Patel |

#### Aim
- To perform Object Detection using SSD

### Contents
1. Section 1 : Loading Datasets
2. Section 2 : Exploratory Data Analysis
3. Section 3 : Utilities for Model Training
4. Section 4 : SSD x MobileNetV3 Training Loop

### File Outcome
*An Object Detection model*

### Library imports

In [ ]:
!pip install torch torchvision -q

In [ ]:
!pip install nbconvert -q

In [ ]:
!pip install clearml -q

In [ ]:
# Import required libraries
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import os
import pandas as pd
import sklearn
import tensorflow as tf
import torch
import torchvision
import keras
import xml.etree.ElementTree as ET

from clearml import Dataset as CMLDataset
from clearml import OutputModel
from clearml import Model
from clearml import Task

from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.ops import box_iou
from torchvision.transforms import functional as F
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from torchvision.models import MobileNet_V3_Large_Weights

In [ ]:
torch.cuda.is_available()
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

### Downloading dataset from ClearML

In [ ]:
dataset_root = CMLDataset.get(
    dataset_name="Object_Detection_Data",
    dataset_project="Assignment 2 : Object Detection"
).get_local_copy()

print(f"Data successfully synced to agent directory: {dataset_root}")

In [ ]:
TRAIN_DIR = os.path.join(dataset_root, 'pascal/train')
VALID_DIR = os.path.join(dataset_root, 'pascal/valid')
TEST_DIR = os.path.join(dataset_root, 'pascal/test')

In [ ]:
CLASSES = ['__background__', 'Ready', 'empty_pod', 'germination', 'pod', 'young']
CLASS_MAP = {
    0: '__background__',
    1: 'Ready',
    2: 'empty_pod',
    3: 'germination',
    4: 'pod',
    5: 'young'
}
COLORS = ['gray', 'lime', 'red', 'cyan', 'yellow', 'magenta']
NUM_CLASSES = len(CLASSES)

### Section 1 : Loading dataset

In [ ]:
# Checking number of images and annotations
train_images = sum(1 for f in os.listdir(TRAIN_DIR) if f.lower().endswith('.jpg'))
valid_images = sum(1 for f in os.listdir(VALID_DIR) if f.lower().endswith('.jpg'))
test_images = sum(1 for f in os.listdir(TEST_DIR) if f.lower().endswith('.jpg'))

train_labels = sum(1 for f in os.listdir(TRAIN_DIR) if f.lower().endswith('.xml'))
valid_labels = sum(1 for f in os.listdir(VALID_DIR) if f.lower().endswith('.xml'))
test_labels = sum(1 for f in os.listdir(TEST_DIR) if f.lower().endswith('.xml'))

print("Number of training images =", train_images)
print("Number of validation images =", valid_images)
print("Number of testing images =", test_images)

print("Number of training labels =", train_labels)
print("Number of validation labels =", valid_labels)
print("Number of testing labels =", test_labels)

*As we have equal number of training, validation, and testing images and labels. Additional manual annotation is not required*

#### Make custom VOC dataset dictionary using images and annotations

In [ ]:
# Function to parce xml files
def parse_voc_xml(xml_file):
    # Parse the XML annotation file using ElementTree
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Initialize lists to store bounding boxes and labels
    boxes, labels = [], []

    # Loop over all object elements in the XML
    for obj in root.findall("object"):
        # Get the object class name
        label = obj.find("name").text

        # Skip labels that are not in the defined CLASSES list
        if label not in CLASSES:
            continue

        # Convert label name to its corresponding index in CLASSES
        labels.append(CLASSES.index(label))

        # Extract the bounding box coordinates from the XML
        bbox = obj.find("bndbox")
        box = [
            float(bbox.find("xmin").text),  # left
            float(bbox.find("ymin").text),  # top
            float(bbox.find("xmax").text),  # right
            float(bbox.find("ymax").text)   # bottom
        ]
        boxes.append(box)

    # Return list of bounding boxes and their corresponding labels
    return boxes, labels

# Class to create custom VOC dataset
class VOCDataset(Dataset):
    def __init__(self, path, transforms=None):
        self.path = path
        self.transforms = transforms
        self.images = [f for f in os.listdir(path) if f.endswith('.jpg')]
        self.xml = [f for f in os.listdir(path) if f.endswith('.xml')]
        self.images.sort()
        self.xml.sort()

    def __getitem__(self, idx):
        img_path = os.path.join(self.path, self.images[idx])
        xml_path = os.path.join(self.path, self.xml[idx])
        img = Image.open(img_path).convert("RGB")

        boxes, labels = parse_voc_xml(xml_path)
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx])
        }
        img = F.to_tensor(img)
        return img, target

    def __len__(self):
        return len(self.images)

#### DataLoader

In [ ]:
dataset = VOCDataset(TRAIN_DIR)

data_loader =  DataLoader(
    dataset,
    batch_size = 8,
    shuffle = True,
    collate_fn = lambda x: tuple(zip(*x)),
    num_workers=4,
    pin_memory=True
)

In [ ]:
len(dataset.images)

In [ ]:
len(dataset.xml)

### Section 2 : Exploring data

In [ ]:
# Function to display image and corresponding bounding boxes
def plot_samples(dataset):
    fig, axes = plt.subplots(1, 3, figsize=(18, 25))
    axes = axes.flatten()
    num_samples = 3
    dataset_size = len(dataset)
    random_indices = np.random.choice(dataset_size, num_samples, replace=False)
    for i in range(num_samples):
        img, target = dataset[random_indices[i]]
        ax = axes[i]
        img = img.permute(1, 2, 0).numpy()
        ax.imshow(img)
        boxes = target['boxes'].numpy()
        labels = target['labels'].numpy()
        for box, label in zip(boxes, labels):
            xmin, ymin, xmax, ymax = box
            width, height = xmax - xmin, ymax - ymin
            # Create a Rectangle patch
            rect = patches.Rectangle(
                (xmin, ymin), width, height,
                linewidth=2, edgecolor=COLORS[label], facecolor='none'
            )
            ax.add_patch(rect)
            ax.text(xmin, ymin, CLASS_MAP[label],
                    bbox=dict(facecolor='red', alpha=0.5), color='white')
        ax.set_title(f"Sample {i}", fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.axis('off')
    plt.show()

In [ ]:
plot_samples(dataset)

### Section 3 : Utilities for Model Training

#### Downloading pre-trained models

In [ ]:
def get_model(num_classes):
    model = ssdlite320_mobilenet_v3_large(
        weights=None,
        weights_backbone="DEFAULT",
        num_classes=num_classes
    )

    for param in model.backbone.parameters():
        param.requires_grad = False
    return model

#### Metric Computation Utility

In [ ]:
def compute_map_ar(preds, targets, num_classes=len(CLASSES)-1):
    # Initialize the results dictionary with default values
    results = {
        'map': 0, 'map_50': 0, 'map_75': 0,
        'map_per_class': torch.zeros(num_classes),
        'mar_1': 0, 'mar_10': 0, 'mar_100': 0,
        'mar_100_per_class': torch.zeros(num_classes),
    }

    # Lists to hold AP and AR values for each class
    aps = [[] for _ in range(num_classes)]
    ars = [[] for _ in range(num_classes)]

    # Loop through each image's predictions and targets
    for pred, target in zip(preds, targets):
        # Loop through each class (excluding background)
        for class_idx in range(1, num_classes+1):
            # Filter boxes by current class
            gt_mask = target['labels'] == class_idx
            pred_mask = pred['labels'] == class_idx

            gt_boxes = target['boxes'][gt_mask]
            pred_boxes = pred['boxes'][pred_mask]
            # Skip if no GT or predictions
            if len(gt_boxes) == 0 and len(pred_boxes) == 0:
                continue

            # Compute IoUs between predictions and ground truth
            ious = torch.zeros((0, 0))
            if len(gt_boxes) > 0 and len(pred_boxes) > 0:
                ious = box_iou(pred_boxes, gt_boxes)

            # Initialize true positives (TP) and matched GT indices
            tp = torch.zeros(len(pred_boxes))
            matched = set()

            # Match predictions to ground truth based on IoU > 0.5
            for i, row in enumerate(ious):
                max_iou, max_j = torch.max(row, dim=0)
                if max_iou > 0.5 and max_j.item() not in matched:
                    tp[i] = 1
                    matched.add(max_j.item())

            # Compute false positives (FP)
            fp = 1 - tp

            # Cumulative TP and FP for precision-recall curve
            cum_tp = torch.cumsum(tp, dim=0)
            cum_fp = torch.cumsum(fp, dim=0)

            # Compute recall and precision
            recalls = cum_tp / (len(gt_boxes) + 1e-6)
            precisions = cum_tp / (cum_tp + cum_fp + 1e-6)

            # Compute AP (area under precision-recall curve)
            ap = torch.trapz(precisions, recalls) if recalls.numel() > 0 else torch.tensor(0.)
            # AR is the max recall value
            ar = recalls[-1] if recalls.numel() > 0 else torch.tensor(0.)

            # Store per-class AP and AR
            aps[class_idx-1].append(ap.item())
            ars[class_idx-1].append(ar.item())

    # Compute average AP and AR for each class
    ap_avg = torch.tensor([np.mean(cls_ap) if cls_ap else 0. for cls_ap in aps])
    ar_avg = torch.tensor([np.mean(cls_ar) if cls_ar else 0. for cls_ar in ars])

    # Save results
    results['map_per_class'] = ap_avg
    results['mar_100_per_class'] = ar_avg
    results['map'] = ap_avg.mean()
    results['map_50'] = ap_avg.mean()
    results['map_75'] = ap_avg.mean()
    results['mar_100'] = ar_avg.mean()
    results['mar_10'] = ar_avg.mean()
    results['mar_1'] = ar_avg.mean()

    return results

#### Evaluation Utility

In [ ]:
def evaluate_map(model, dataset, iou_thresholds = [0.5, 0.75]):
    model.eval()
    all_preds, all_targets = [], []
    for img, target in dataset:
        img = img.to(device, non_blocking=True).unsqueeze(0)
        with torch.no_grad():
            pred = model(img)[0]

        keep = pred['scores'] > 0.05
        pred_boxes = pred['boxes'][keep].cpu()
        pred_labels = pred['labels'][keep].cpu()
        pred_scores = pred['scores'][keep].cpu()

        all_preds.append({
            'boxes': pred_boxes,
            'labels': pred_labels,
            'scores': pred_scores
        })

        all_targets.append({
            'boxes': target['boxes'].cpu(),
            'labels': target['labels'].cpu()
        })

        return compute_map_ar(all_preds, all_targets)

#### Training setup

In [ ]:
model = get_model(NUM_CLASSES).to(device, non_blocking=True)

In [ ]:
# Define the optimizer
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.0005,
    momentum=0.9,
    weight_decay=0.0005
)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

# Number of epochs to train
num_epochs = 300
start_epoch = 0

# Logging and checkpointing settings
print_every = 100      # Print loss every N batches
save_every = 5     # Save model every N epochs
val_every = 5        # Validate every N epochs

# Initialize logging lists
epoch_losses = []
iteration_losses = []
val_maps = []

val_dataset = VOCDataset(VALID_DIR)


### Section 4 : SSD Training Loop

In [ ]:
class ClearMLEarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
            print(f"Initial baseline validation loss set to: {val_loss:.4f}")
            return

        target_improvement = self.best_loss * 0.30

        if val_loss <= target_improvement:
            print(f"--> Excellent progress! Loss dropped by >30% (Old Best: {self.best_loss:.4f} -> New Best: {val_loss:.4f}). Resetting patience.")
            self.best_loss = val_loss
            self.counter = 0
        else:
            if val_loss < self.best_loss:
                self.best_loss = val_loss
            self.counter += 1
            print(f"EarlyStopping counter: {self.counter} out of {self.patience}")
        if self.counter >= self.patience:
            self.early_stop = True

In [ ]:
output_model = OutputModel(
    task=task,
    framework="PyTorch"
)

early_stopper = ClearMLEarlyStopping(patience=30)

for epoch in range(start_epoch, num_epochs):
    model.train()  # Set model to training mode
    total_loss = 0  # Track total loss for the epoch

    for i, (images, targets) in enumerate(data_loader):
        # Move all images and targets to the selected device
        images = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

        # Get the loss dict from the model
        loss_dict = model(images, targets)

        # Combine all losses into a single scalar
        losses = sum(loss for loss in loss_dict.values())

        # Backward pass and optimizer step
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

        # Accumulate batch loss and append to losses per iteration
        total_loss += losses.item()
        iteration_losses.append(losses.item())
        # Print batch loss every few iterations
        if (i + 1) % print_every == 0:
            print(f"  [Epoch {epoch+1}, Iter {i+1}] Loss: {losses.item():.4f}")

    # Print total loss at the end of the epoch and store losses for loss curve
    print(f"Epoch [{epoch+1}/{num_epochs}], Total Loss: {total_loss:.4f}")
    epoch_losses.append(total_loss)

    if (epoch+1) % val_every == 0:
        # Evaluate on validation set
        val_results = evaluate_map(model, val_dataset)
        val_map = val_results["map"].item()
        val_maps.append(val_map)
        print(f"Validation mAP at epoch {epoch+1}: {val_map:.4f}")

    checkpoint_filename = f"ssdlite_mobilenet_epoch_{epoch+1}.pth"
    # Save checkpoint for loss every few epochs
    if (epoch + 1) % save_every == 0:
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
        }, checkpoint_filename)
        output_model.update_weights(
            weights_filename=checkpoint_filename,
            auto_delete_previous=True
        )
        print(f"Epoch {epoch+1}: Checkpoint successfully backed up to ClearML storage.")

    # Early stopping in ClearML
    early_stopper(losses.item())
    if early_stopper.early_stop:
        print(f"Early stopping triggered at epoch {epoch}. Training halted.")
        break

# save final model after training is complete
torch.save({
    'epoch': epoch,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
}, checkpoint_filename)
output_model.update_weights(
    weights_filename=checkpoint_filename,
)
print(f"Epoch {epoch+1}: Checkpoint successfully backed up to ClearML storage.")

In [ ]:
logger = task.get_logger()

In [ ]:
# Plot training loss per epoch
plt.figure(figsize=(10, 4))
plt.plot(epoch_losses, marker='o')
plt.title("Training Loss per Epoch")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.show()
logger.report_matplotlib_figure(
    title="SSD training loss",
    series="Line Plot",
    figure=plt.gcf(),
    iteration=0
)

In [ ]:
# Plot validation mAP per epoch
fig = plt.figure(figsize=(10, 4))
fig.plot(val_maps, marker='s', color='green')
fig.title("Validation mAP per Epoch")
fig.xlabel("Epoch")
fig.ylabel("mAP (IoU=1.0)")
fig.grid(True)
fig.show()
logger.report_matplotlib_figure(
    title="SSD mAP Validation",
    series="Line Plot",
    figure=plt.gcf(),
    iteration=0
)

In [ ]:
train_results = evaluate_map(model, dataset)

# Print overall mAP/mAR results for the training set
print("Train set mAP/mAR results:")
for k, v in train_results.items():
    # Convert torch tensors to NumPy arrays for clean printing
    if isinstance(v, torch.Tensor):
        print(f"{k}: {v.numpy()}")
    else:
        print(f"{k}: {v}")

In [ ]:
test_dataset = VOCDataset(TEST_DIR)

# Create a DataLoader for the test set
test_loader = DataLoader(
    test_dataset,
    batch_size=2,
    shuffle=False,
    collate_fn=lambda x: tuple(zip(*x))
)

In [ ]:
# Evaluate the model on the test dataset
test_results = evaluate_map(model, test_dataset)

# Print overall mAP/mAR results for the test set
print("Test set mAP/mAR results:")
for k, v in test_results.items():
    # Convert torch tensors to NumPy arrays for cleaner display
    if isinstance(v, torch.Tensor):
        print(f"{k}: {v.numpy()}")
    else:
        print(f"{k}: {v}")

In [ ]:
# Force Jupyter to save the current notebook state to disk
from IPython.display import display, Javascript
display(Javascript('IPython.notebook.save_checkpoint();'))

# Fetch the current task and upload the executed file
task = Task.current_task()
if task:
    task.upload_artifact(name='executed_notebook', artifact_object='Assignment_2_OD_SSD.ipynb')